# 🔄 Customer Churn Analysis
**Author:** Ahmed Walid  
**Tools:** Python · Pandas · Scikit-learn · Matplotlib · Seaborn  
**Goal:** Predict which customers are likely to churn using Machine Learning.


## 1. 📦 Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 5)
sns.set_theme(style='darkgrid', palette='muted')
print('✅ Libraries loaded successfully')

## 2. 🗄️ Generate Dataset

In [ ]:
np.random.seed(42)
n = 1000

df = pd.DataFrame({
    'CustomerID':        [f'CUST-{i:04d}' for i in range(1, n+1)],
    'Age':               np.random.randint(18, 70, n),
    'Gender':            np.random.choice(['Male', 'Female'], n),
    'Tenure':            np.random.randint(1, 72, n),
    'MonthlyCharges':    np.round(np.random.uniform(20, 120, n), 2),
    'TotalCharges':      np.round(np.random.uniform(100, 8000, n), 2),
    'Contract':          np.random.choice(['Month-to-month', 'One year', 'Two year'], n, p=[0.5, 0.3, 0.2]),
    'InternetService':   np.random.choice(['DSL', 'Fiber optic', 'No'], n, p=[0.4, 0.4, 0.2]),
    'TechSupport':       np.random.choice(['Yes', 'No'], n),
    'PaymentMethod':     np.random.choice(['Electronic check', 'Mailed check', 'Bank transfer', 'Credit card'], n),
    'NumProducts':       np.random.randint(1, 6, n),
    'Complains':         np.random.randint(0, 5, n),
})

# Churn logic: higher chance if month-to-month, high charges, low tenure
churn_prob = (
    (df['Contract'] == 'Month-to-month').astype(int) * 0.3 +
    (df['MonthlyCharges'] > 80).astype(int) * 0.2 +
    (df['Tenure'] < 12).astype(int) * 0.2 +
    (df['Complains'] > 2).astype(int) * 0.2 +
    np.random.uniform(0, 0.1, n)
)
df['Churn'] = (churn_prob > 0.4).astype(int)

print(f'✅ Dataset: {df.shape[0]:,} rows × {df.shape[1]} columns')
print(f'   Churn Rate: {df["Churn"].mean()*100:.1f}%')
df.head()

## 3. 🔍 Exploratory Data Analysis (EDA)

In [ ]:
print('=== Dataset Info ===')
print(df.dtypes)
print(f'\nMissing values: {df.isnull().sum().sum()}')

### 3.1 Churn Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Count plot
churn_counts = df['Churn'].value_counts()
axes[0].bar(['No Churn', 'Churn'], churn_counts.values, color=['#1976D2', '#EF5350'])
axes[0].set_title('Churn Distribution', fontsize=14, fontweight='bold')
for i, v in enumerate(churn_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# Pie chart
axes[1].pie(churn_counts.values, labels=['No Churn', 'Churn'],
            colors=['#1976D2', '#EF5350'], autopct='%1.1f%%', startangle=90)
axes[1].set_title('Churn Rate', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('churn_distribution.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: churn_distribution.png')

### 3.2 Churn by Contract Type

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

contract_churn = df.groupby('Contract')['Churn'].mean() * 100
axes[0].bar(contract_churn.index, contract_churn.values, color=['#EF5350', '#FFA726', '#66BB6A'])
axes[0].set_title('Churn Rate by Contract Type', fontsize=13, fontweight='bold')
axes[0].set_ylabel('Churn Rate (%)')
for i, v in enumerate(contract_churn.values):
    axes[0].text(i, v + 0.5, f'{v:.1f}%', ha='center', fontweight='bold')

tenure_churn = df.groupby(pd.cut(df['Tenure'], bins=[0,12,24,36,48,72]))['Churn'].mean() * 100
axes[1].bar(range(len(tenure_churn)), tenure_churn.values, color='#42A5F5')
axes[1].set_xticks(range(len(tenure_churn)))
axes[1].set_xticklabels(['0-12m', '12-24m', '24-36m', '36-48m', '48-72m'])
axes[1].set_title('Churn Rate by Tenure', fontsize=13, fontweight='bold')
axes[1].set_ylabel('Churn Rate (%)')

plt.tight_layout()
plt.savefig('churn_by_contract_tenure.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: churn_by_contract_tenure.png')

### 3.3 Monthly Charges vs Churn

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color in zip([0, 1], ['#1976D2', '#EF5350']):
    axes[0].hist(df[df['Churn']==label]['MonthlyCharges'],
                 bins=30, alpha=0.6, color=color,
                 label='No Churn' if label==0 else 'Churn')
axes[0].set_title('Monthly Charges Distribution by Churn', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Monthly Charges ($)')
axes[0].legend()

sns.boxplot(data=df, x='Churn', y='MonthlyCharges',
            palette={0:'#1976D2', 1:'#EF5350'}, ax=axes[1])
axes[1].set_xticklabels(['No Churn', 'Churn'])
axes[1].set_title('Monthly Charges Boxplot', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('charges_vs_churn.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: charges_vs_churn.png')

### 3.4 Correlation Heatmap

In [ ]:
num_cols = ['Age', 'Tenure', 'MonthlyCharges', 'TotalCharges', 'NumProducts', 'Complains', 'Churn']
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Heatmap', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: correlation_heatmap.png')

## 4. 🤖 Machine Learning

### 4.1 Preprocessing

In [ ]:
df_ml = df.drop('CustomerID', axis=1).copy()

le = LabelEncoder()
cat_cols = ['Gender', 'Contract', 'InternetService', 'TechSupport', 'PaymentMethod']
for col in cat_cols:
    df_ml[col] = le.fit_transform(df_ml[col])

X = df_ml.drop('Churn', axis=1)
y = df_ml['Churn']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'✅ Train: {X_train.shape[0]} samples | Test: {X_test.shape[0]} samples')

### 4.2 Logistic Regression

In [ ]:
lr = LogisticRegression(random_state=42, max_iter=1000)
lr.fit(X_train_sc, y_train)
y_pred_lr = lr.predict(X_test_sc)

print('=== Logistic Regression ===')
print(classification_report(y_test, y_pred_lr, target_names=['No Churn', 'Churn']))
print(f'ROC-AUC: {roc_auc_score(y_test, lr.predict_proba(X_test_sc)[:,1]):.3f}')

### 4.3 Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print('=== Random Forest ===')
print(classification_report(y_test, y_pred_rf, target_names=['No Churn', 'Churn']))
print(f'ROC-AUC: {roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]):.3f}')

### 4.4 Model Comparison & Confusion Matrices

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, title, color in zip(
    axes,
    [y_pred_lr, y_pred_rf],
    ['Logistic Regression', 'Random Forest'],
    ['Blues', 'Reds']
):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap=color, ax=ax,
                xticklabels=['No Churn','Churn'],
                yticklabels=['No Churn','Churn'])
    ax.set_title(f'{title} — Confusion Matrix', fontsize=13, fontweight='bold')
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig('confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: confusion_matrices.png')

### 4.5 ROC Curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for model, X_input, name, color in [
    (lr, X_test_sc, 'Logistic Regression', '#1976D2'),
    (rf, X_test,    'Random Forest',        '#EF5350')
]:
    fpr, tpr, _ = roc_curve(y_test, model.predict_proba(X_input)[:,1])
    auc = roc_auc_score(y_test, model.predict_proba(X_input)[:,1])
    ax.plot(fpr, tpr, color=color, linewidth=2, label=f'{name} (AUC = {auc:.3f})')

ax.plot([0,1],[0,1],'k--', linewidth=1)
ax.set_title('ROC Curve Comparison', fontsize=15, fontweight='bold')
ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
ax.legend(fontsize=12)
plt.tight_layout()
plt.savefig('roc_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: roc_curve.png')

### 4.6 Feature Importance (Random Forest)

In [ ]:
importance = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
importance.plot(kind='barh', color='#1976D2', ax=ax)
ax.set_title('Feature Importance — Random Forest', fontsize=15, fontweight='bold')
ax.set_xlabel('Importance Score')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: feature_importance.png')

## 5. 📌 Summary & Recommendations

In [ ]:
lr_auc = roc_auc_score(y_test, lr.predict_proba(X_test_sc)[:,1])
rf_auc = roc_auc_score(y_test, rf.predict_proba(X_test)[:,1])

print('=' * 45)
print('       🔄 CHURN ANALYSIS SUMMARY')
print('=' * 45)
print(f'  📊 Dataset Size       : {len(df):,} customers')
print(f'  📉 Churn Rate         : {df["Churn"].mean()*100:.1f}%')
print(f'  🤖 Logistic Reg AUC   : {lr_auc:.3f}')
print(f'  🌲 Random Forest AUC  : {rf_auc:.3f}')
print(f'  🏆 Best Model         : {"Random Forest" if rf_auc > lr_auc else "Logistic Regression"}')
print('=' * 45)
print()
print('📌 Key Drivers of Churn:')
top3 = importance.sort_values(ascending=False).head(3)
for i, (feat, val) in enumerate(top3.items(), 1):
    print(f'   {i}. {feat} ({val:.3f})')

## 6. 💾 Export Data

In [ ]:
df.to_csv('customer_churn_data.csv', index=False)
print(f'✅ Exported → customer_churn_data.csv ({len(df):,} rows)')

## 7. 📝 Conclusion

### Key Findings:
- **Month-to-month** contracts have the highest churn rate (~60%)
- Customers with **tenure < 12 months** are most at risk
- **High monthly charges** strongly correlate with churn
- **Random Forest** outperforms Logistic Regression on this dataset

### Recommendations:
1. Offer loyalty discounts to customers in their first year
2. Incentivize contract upgrades from month-to-month to annual plans
3. Review pricing strategy for high-charge customer segments
